<a href="https://colab.research.google.com/github/Mohamed-Mohamed-Ibrahim/Code-Generation-and-Guarding/blob/SFT/sft/peft-sft-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!apt install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tree is already the newest version (2.0.2-1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [31]:
!pip install peft transformers[torch] trl bitsandbytes datasets wandb -U --q

In [32]:
import os
import random, math
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training

In [33]:
import wandb
wandb.login()

True

In [46]:
os.environ["WANDB_PROJECT"] = "sft_nlp_lab"  # name your W&B project
os.environ["WANDB_LOG_MODEL"] = "checkpoint"  # log all model checkpoints

In [34]:
dataset = load_dataset("flytech/python-codes-25k",split="train")

In [35]:
dataset

Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

In [36]:
shuffled_dataset = dataset.shuffle(seed=42)
train_dataset = shuffled_dataset.select(range(30))
eval_dataset = shuffled_dataset.select(range(2))
test_dataset = shuffled_dataset.select(range(2))

In [37]:
print("="*100)
print(shuffled_dataset)
print("="*100)
print(shuffled_dataset[0]['input'])
print("="*100)
print(shuffled_dataset[0]['instruction'])
print("="*100)
print(shuffled_dataset[0]['text'])
print("="*100)
print(shuffled_dataset[0]['output'])
print("="*100)


Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order
Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order Let's roll! The ball's in our court! ```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```
```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```


In [38]:
# HF hub ID
BASE_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
BASE_MODEL_NAME = "arnir0/Tiny-LLM"

# Push adapter artifacts post training
OUTPUT_DIR = "./qlora-peft-output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")

## Hyperparameters

In [39]:
# Lab
lora_r = 16
lora_target_modules = "all-linear"
bs =  1
gradient_accumulation_steps = 4
epochs = 1
lr = 2e-4
optim = "paged_adamw_8bit"
max_length = 1024

# Others

In [40]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [41]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [42]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    # device_map={"": torch.cuda.current_device()},
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

In [43]:
lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=16,
    target_modules=lora_target_modules,  # Llama-style
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [49]:
from trl import SFTConfig, SFTTrainer

# Define ALL configuration parameters within SFTConfig
config = SFTConfig(
    # --- Standard TrainingArguments parameters ---
    output_dir=OUTPUT_DIR,
    learning_rate=lr,
    num_train_epochs=epochs,
    per_device_train_batch_size=bs,
    gradient_accumulation_steps=gradient_accumulation_steps,
    save_strategy="epoch",
    bf16=True,
    optim=optim,

    # --- SFT-specific parameters (moved here from previous iterations) ---
    dataset_text_field="text",
    max_length=max_length,
    packing=False,

    # Scheduler
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Compute loss on model answers only
    completion_only_loss=True,

    # Logging
    logging_steps=50,
    report_to="wandb", # enables logging to W&B 😎
)

# Initialize the SFTTrainer, passing the SFTConfig object to the 'args' parameter
trainer = SFTTrainer(
    model=model,
    args=config,                # SFTConfig object
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [50]:
trainer.train()

Step,Training Loss


wandb: Adding directory to artifact (qlora-peft-output/checkpoint-8)... Done. 0.0s


TrainOutput(global_step=8, training_loss=6.324220657348633, metrics={'train_runtime': 4.9792, 'train_samples_per_second': 6.025, 'train_steps_per_second': 1.607, 'total_flos': 203092411392.0, 'train_loss': 6.324220657348633, 'entropy': 5.736804755528768, 'num_tokens': 4888.0, 'mean_token_accuracy': 0.13997484222054482, 'epoch': 1.0})

In [51]:
ADAPTER_DIR = "./qlora-peft-output/adapter"

# 6. Save adapter weights

In [52]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")

Adapter saved to: ./qlora-peft-output/adapter


In [53]:
!tree


.
├── merged-model
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   └── tokenizer.model
├── qlora-peft-output
│   ├── adapter
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── README.md
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   └── tokenizer.model
│   ├── checkpoint-3
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│   │   ├── scheduler.pt
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   ├── tokenizer.model
│   │   ├── trainer_state.json
│   │   └── training_args.bin
│   ├── checkpoint-8
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│

In [55]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE_ID = "Qwen/Qwen2-1.5B-Instruct"
BASE_ID = "arnir0/Tiny-LLM"
ADAPTER_DIR = "./qlora-peft-output/adapter"
MERGED_DIR = "./merged-model"

# -------------------------------------------------------
# Load tokenizer
# -------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------------------------------
# Load base model in fp16/bf16
# (merged model will be full precision LLM)
# -------------------------------------------------------
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# -------------------------------------------------------
# Attach adapter
# -------------------------------------------------------
print("Loading adapter onto base model...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# -------------------------------------------------------
# Merge adapter → base model
# -------------------------------------------------------
print("Merging LoRA adapter into base model weights...")
merged_model = model.merge_and_unload()   # <-- key line

# -------------------------------------------------------
# Save merged full model (optional)
# -------------------------------------------------------
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\nMerged model saved to: {MERGED_DIR}\n")

Loading base model...
Loading adapter onto base model...
Merging LoRA adapter into base model weights...

Merged model saved to: ./merged-model



In [56]:
!tree

.
├── merged-model
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   └── tokenizer.model
├── qlora-peft-output
│   ├── adapter
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── README.md
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   └── tokenizer.model
│   ├── checkpoint-3
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│   │   ├── scheduler.pt
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   ├── tokenizer.model
│   │   ├── trainer_state.json
│   │   └── training_args.bin
│   ├── checkpoint-8
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│

# Run merged-model inference

In [57]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [59]:
print("==================== MERGED MODEL OUTPUT ====================\n")

for p in test_dataset:
    wrapped = (
        "You are a personalized assistant that knows details about the user based "
        "on prior fine-tuning data.\n\n"
        f"Question: {p['instruction']}\nAnswer:"
    )

    print(f"Output: {p['output']}\n")
    output = generate(wrapped)
    print(output)
    print("\n", "-" * 120, "\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


==================== MERGED MODEL OUTPUT ====================

Output: ```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order
Answer: 100%
Question: What is the number of the number of the number of
Answer: 100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

 ------------------------------------------------------------------------------------------------------------------------ 

Output: ```python
def is_prime(num):
    if num < 2:
        return False
    if num == 2 or num == 3: 
        return True

    i = 2
    while i*i <= num: 
        if (num % i) == 0: 
            return False
        i += 1

    return True
```

You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Optimize the following Python code to make it run f

In [60]:
trainer.evaluate()

{'eval_loss': 6.241007328033447,
 'eval_runtime': 0.0341,
 'eval_samples_per_second': 58.629,
 'eval_steps_per_second': 29.315,
 'eval_entropy': 5.337177753448486,
 'eval_num_tokens': 4888.0,
 'eval_mean_token_accuracy': 0.19387754797935486,
 'epoch': 1.0}

In [61]:
metrics = trainer.evaluate()
ppl = math.exp(metrics['eval_loss']) if metrics['eval_loss'] < 20 else float('inf')
print('KL-SFT post-train eval_loss:', metrics['eval_loss'], 'ppl:', ppl)
print('Eval losses over steps:', [(h.get('step'), h.get('eval_loss')) for h in trainer.state.log_history if 'eval_loss' in h])


KL-SFT post-train eval_loss: 6.241007328033447 ppl: 513.3753879867551
Eval losses over steps: [(8, 6.241007328033447), (8, 6.241007328033447)]


# Resources

- https://wandb.ai/capecape/alpaca_ft/reports/How-to-Fine-tune-an-LLM-Part-3-The-HuggingFace-Trainer--Vmlldzo1OTEyNjMy
- https://youtu.be/t1caDsMzWBk